<a href="https://colab.research.google.com/github/AndreiMoraru123/learning_ray/blob/main/notebooks/ch_06_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Processing with Ray


You can run this notebook directly in
[Colab](https://colab.research.google.com/github/maxpumperla/learning_ray/blob/main/notebooks/ch_06_data_processing.ipynb).
<a target="_blank" href="https://colab.research.google.com/github/maxpumperla/learning_ray/blob/main/notebooks/ch_06_data_processing.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For this chapter you need to install the following dependencies:

In [1]:
! pip install "ray[data]"
! pip install "scikit-learn"
! pip install "dask"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 10.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1



To import utility files for this chapter, on Colab you will also have to clone
the repo and copy the code files to the base path of the runtime:

In [2]:
!git clone https://github.com/maxpumperla/learning_ray
%cp -r learning_ray/notebooks/* .

Cloning into 'learning_ray'...
remote: Enumerating objects: 1385, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 1385 (delta 9), reused 5 (delta 5), pack-reused 1371 (from 2)
Receiving objects: 100% (1385/1385), 119.79 MiB | 17.15 MiB/s, done.
Resolving deltas: 100% (753/753), done.
Updating files: 100% (87/87), done.


![Simple Ray Data](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/AIR_data.png)


![Data Pipeline 1](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/data_pipeline_1.png)

![Data Pipeline 2](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/data_pipeline_2.png)

![Data Positioning 1](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/data_positioning_1.png)

![Data Positioning 2](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/data_positioning_2.png)

![Data Architecture](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/datasets_arch.png)

![Data ML Workflow](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_06/ml_workflow.png)

In [3]:
import ray

# Create a dataset containing integers in the range [0, 10000).
ds = ray.data.range(10000)

# Basic operations: show the size of the dataset, get a few samples, print the schema.
print(ds.count())  # -> 10000
print(ds.take(5))  # -> [0, 1, 2, 3, 4]
print(ds.schema())  # -> <class 'int'>

2025-12-07 08:34:37,176	INFO worker.py:2023 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2025-12-07 08:35:00,086	INFO dataset.py:3485 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2025-12-07 08:35:00,151	INFO logging.py:397 -- Registered dataset logger for dataset dataset_1_0
2025-12-07 08:35:00,173	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:35:00,174	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange] ->

10000


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

- limit=5 2: 0.00 row [00:00, ? row/s]

2025-12-07 08:35:00,277	WARNING resource_manager.py:136 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(raylet) [2025-12-07 08:35:06,239 E 1513 1513] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
2025-12-07 08:35:10,078	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_1_0 execution finished in 9.90 seconds
2025-12-07 08:35:10,119	INFO util.py:257 -- Exiting prefetcher's background thread


[{'id': 0}, {'id': 1}, {'id': 2}, {'id': 3}, {'id': 4}]
Column  Type
------  ----
id      int64


In [4]:
# Save the dataset to a local file and load it back.
ray.data.range(10000).write_csv("local_dir")
ds = ray.data.read_csv("local_dir")
print(ds.count())

2025-12-07 08:35:10,585	INFO logging.py:397 -- Registered dataset logger for dataset dataset_4_0
2025-12-07 08:35:10,598	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_4_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:35:10,600	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange->Write]


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange->Write 1: 0.00 row [00:00, ? row/s]

2025-12-07 08:35:10,937	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_4_0 execution finished in 0.34 seconds
2025-12-07 08:35:11,127	INFO dataset.py:5193 -- Data sink CSV finished. 10000 rows and 78.1KiB data written.
2025-12-07 08:35:11,160	INFO logging.py:397 -- Registered dataset logger for dataset dataset_7_0
2025-12-07 08:35:11,177	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:35:11,180	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV->Project] -> AggregateNumRows[AggregateNumRows]


Running 0:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

- ReadCSV->Project 1: 0.00 row [00:00, ? row/s]

- AggregateNumRows 2:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

2025-12-07 08:35:11,921	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_7_0 execution finished in 0.74 seconds
2025-12-07 08:35:11,937	INFO util.py:257 -- Exiting prefetcher's background thread


10000


In [9]:
ds1 = ray.data.range(10000)
ds2 = ray.data.range(10000)
ds3 = ds1.union(ds2)
print(ds3.count())  # -> 20000

# Filter the combined dataset to only the even elements.
ds3 = ds3.filter(lambda x: x['id'] % 2 == 0)
print(ds3.count())  # -> 10000
print(ds3.take(5))  # -> [{'id': 0}, {'id': 2}, {'id': 4}, {'id': 6}, {'id': 8}]

# Sort the filtered dataset.
ds3 = ds3.sort('id')  # <3>
print(ds3.take(5))  # -> [{'id': 0}, {'id': 0}, {'id': 2}, {'id': 2}, {'id': 4}]

2025-12-07 08:38:34,307	INFO logging.py:397 -- Registered dataset logger for dataset dataset_38_0
2025-12-07 08:38:34,324	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_38_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:38:34,325	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_38_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange], InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange] -> UnionOperator[UnionOperator(ReadRange, ReadRange)] -> TaskPoolMapOperator[Project] -> AggregateNumRows[AggregateNumRows]


Running 0:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

- ReadRange 2: 0.00 row [00:00, ? row/s]

- UnionOperator(ReadRange, ReadRange) 3: 0.00 row [00:00, ? row/s]

...

Shuffle Map 7:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

Shuffle Reduce 8:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

- limit=5 9: 0.00 row [00:00, ? row/s]

2025-12-07 08:38:39,517	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_43_0 execution finished in 1.89 seconds
2025-12-07 08:38:39,589	INFO util.py:257 -- Exiting prefetcher's background thread


[{'id': 0}, {'id': 0}, {'id': 2}, {'id': 2}, {'id': 4}]


In [12]:
ds = ray.data.range(10)
print(ds.schema())
print(ds.take(3))

2025-12-07 08:39:19,948	INFO logging.py:397 -- Registered dataset logger for dataset dataset_48_0
2025-12-07 08:39:19,955	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_48_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:39:19,956	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_48_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange] -> LimitOperator[limit=3]


Column  Type
------  ----
id      int64


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

- limit=3 2: 0.00 row [00:00, ? row/s]

2025-12-07 08:39:20,181	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_48_0 execution finished in 0.22 seconds
2025-12-07 08:39:20,213	INFO util.py:257 -- Exiting prefetcher's background thread


[{'id': 0}, {'id': 1}, {'id': 2}]


In [13]:
ds1 = ray.data.range(10000).materialize()
print(ds1.num_blocks())  # -> 200
ds2 = ray.data.range(10000).materialize()
print(ds2.num_blocks())  # -> 200
ds3 = ds1.union(ds2).materialize()
print(ds3.num_blocks())  # -> 400

print(ds3.repartition(200).materialize().num_blocks())  # -> 200

2025-12-07 08:39:20,721	INFO logging.py:397 -- Registered dataset logger for dataset dataset_50_0
2025-12-07 08:39:20,728	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_50_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:39:20,729	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_50_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange]


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

2025-12-07 08:39:20,891	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_50_0 execution finished in 0.16 seconds
2025-12-07 08:39:20,951	INFO logging.py:397 -- Registered dataset logger for dataset dataset_53_0
2025-12-07 08:39:20,959	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_53_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:39:20,959	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_53_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange]


4


...

Running 0: 0.00 row [00:00, ? row/s]

- Repartition 1: 0.00 row [00:00, ? row/s]

Split Repartition 2:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

2025-12-07 08:39:22,334	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_59_0 execution finished in 0.96 seconds
2025-12-07 08:39:22,388	WARNING plan.py:504 -- Warning: The Ray cluster currently does not have any available CPUs. The Dataset job will hang unless more CPUs are freed up. A common reason is that cluster resources are used by Actors or Tune trials; see the following link for more details: https://docs.ray.io/en/latest/data/data-internals.html#ray-data-and-tune


200


In [14]:
ds = ray.data.from_items([{"id": "abc", "value": 1}, {"id": "def", "value": 2}])
print(ds.schema())  # -> id: string, value: int64

Column  Type
------  ----
id      string
value   int64


In [15]:
pandas_df = ds.to_pandas()  # pandas_df will inherit the schema from our Dataset.

2025-12-07 08:40:22,197	INFO logging.py:397 -- Registered dataset logger for dataset dataset_61_0


Running 0: 0.00 row [00:00, ? row/s]

2025-12-07 08:40:22,263	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_61_0 execution finished in 0.00 seconds
2025-12-07 08:40:22,276	INFO util.py:257 -- Exiting prefetcher's background thread


In [19]:
ds = ray.data.range(10000).map(lambda x: {'squared_id': x['id'] ** 2})
ds.take(5)  # -> [{'squared_id': 0}, {'squared_id': 1}, {'squared_id': 4}, {'squared_id': 9}, {'squared_id': 16}]

2025-12-07 08:41:44,392	INFO logging.py:397 -- Registered dataset logger for dataset dataset_73_0
2025-12-07 08:41:44,402	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_73_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:41:44,404	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_73_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange] -> LimitOperator[limit=5] -> TaskPoolMapOperator[Map(<lambda>)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

- limit=5 2: 0.00 row [00:00, ? row/s]

- Map(<lambda>) 3: 0.00 row [00:00, ? row/s]

2025-12-07 08:41:44,723	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_73_0 execution finished in 0.32 seconds
2025-12-07 08:41:44,752	INFO util.py:257 -- Exiting prefetcher's background thread


[{'squared_id': 0},
 {'squared_id': 1},
 {'squared_id': 4},
 {'squared_id': 9},
 {'squared_id': 16}]

In [21]:
import numpy as np

ds = ray.data.range(10000).map_batches(lambda batch: {'squared_id': np.square(batch['id']).tolist()})
ds.take(5)  # -> [{'squared_id': 0}, {'squared_id': 1}, {'squared_id': 4}, {'squared_id': 9}, {'squared_id': 16}]

2025-12-07 08:42:22,460	INFO logging.py:397 -- Registered dataset logger for dataset dataset_79_0
2025-12-07 08:42:22,472	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_79_0. Full logs are in /tmp/ray/session_2025-12-07_08-34-22_639845_1005/logs/ray-data
2025-12-07 08:42:22,474	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_79_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange] -> LimitOperator[limit=5] -> TaskPoolMapOperator[MapBatches(<lambda>)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

- limit=5 2: 0.00 row [00:00, ? row/s]

- MapBatches(<lambda>) 3: 0.00 row [00:00, ? row/s]

2025-12-07 08:42:22,905	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_79_0 execution finished in 0.43 seconds


[{'squared_id': 0},
 {'squared_id': 1},
 {'squared_id': 4},
 {'squared_id': 9},
 {'squared_id': 16}]

2025-12-07 08:42:22,937	INFO util.py:257 -- Exiting prefetcher's background thread


In [22]:
def load_model():
    # Returns a dummy model for this example.
    # In reality, this would likely load some model weights onto a GPU.
    class DummyModel:
        def __call__(self, batch):
            return batch

    return DummyModel()


class MLModel:
    def __init__(self):
        # load_model() will only run once per actor that's started.
        self._model = load_model()

    def __call__(self, batch):
        return self._model(batch)


ds.map_batches(MLModel, compute="actors")


cpu_intensive_preprocessing = lambda batch: batch
gpu_intensive_inference = lambda batch: batch

In [ ]:
# NOTE: this only works if you create an S3 bucket and upload the data there.
ds = (ray.data.read_parquet("s3://my_bucket/input_data")
      .map(cpu_intensive_preprocessing)
      .map_batches(gpu_intensive_inference, compute="actors", num_gpus=1)
      .repartition(10))

ds.write_parquet("s3://my_bucket/output_predictions")

In [ ]:
# NOTE: this only works if you create an S3 bucket and upload the data there.
ds = (ray.data.read_parquet("s3://my_bucket/input_data")
      .window(blocks_per_window=5)
      .map(cpu_intensive_preprocessing)
      .map_batches(gpu_intensive_inference, compute="actors", num_gpus=1)
      .repartition(10))
ds.write_parquet("s3://my_bucket/output_predictions")

In [44]:
from sklearn import datasets
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
import numpy as np


@ray.remote
class TrainingWorker:
    def __init__(self, alpha: float):
        self._model = SGDClassifier(alpha=alpha)

    def train(self, train_shard: ray.data.Dataset, num_epochs: int = 1, shuffle_each_window: bool = False):
        for _ in range(num_epochs):
            X_epoch = []
            Y_epoch = []
            # Iterate through batches of the shard for the current epoch
            for batch in train_shard.iter_batches(batch_size=256):
                # Assuming the dataset items are tuples (X_row, Y_label) stored under an 'item' key
                items_in_batch = batch['item']
                X_batch, Y_batch = zip(*items_in_batch)
                X_epoch.extend(X_batch)
                Y_epoch.extend(Y_batch)

            # Fit the model with all data collected for the current epoch
            self._model.partial_fit(np.array(X_epoch), np.array(Y_epoch), classes=[0, 1])

        return self._model

    def test(self, X_test: np.ndarray, Y_test: np.ndarray):
        return self._model.score(X_test, Y_test)

In [45]:
ALPHA_VALS = [0.00008, 0.00009, 0.0001, 0.00011, 0.00012]

print(f"Starting {len(ALPHA_VALS)} training workers.")
workers = [TrainingWorker.remote(alpha) for alpha in ALPHA_VALS]

Starting 5 training workers.


In [46]:
X_train, X_test, Y_train, Y_test = train_test_split(
    *datasets.make_classification()
)

train_ds = ray.data.from_items(list(zip(X_train, Y_train)))
shards = (train_ds
          .split(len(workers), locality_hints=workers))

ray.get([worker.train.remote(shard, num_epochs=10) for worker, shard in zip(workers, shards)])

(pid=6328) Running 0: 0.00 row [00:00, ? row/s]

(TrainingWorker pid=6328) Registered dataset logger for dataset dataset_120_0
(TrainingWorker pid=6328) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bars = True` and `ray.data.DataContext.get_current().use_ray_tqdm = False`.
(TrainingWorker pid=6328) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(TrainingWorker pid=6328) ✔️  Dataset dataset_120_0 execution finished in 0.00 seconds
(TrainingWorker pid=6328) Exiting prefetcher's background thread


(pid=6328) Running 0: 0.00 row [00:00, ? row/s]

(TrainingWorker pid=6328) Registered dataset logger for dataset dataset_120_1
(TrainingWorker pid=6328) ✔️  Dataset dataset_120_1 execution finished in 0.00 seconds


(pid=6328) Running 0: 0.00 row [00:00, ? row/s]

...

(pid=6502) Running 0: 0.00 row [00:00, ? row/s]

(pid=6502) Running 0: 0.00 row [00:00, ? row/s]

(TrainingWorker pid=6328) [2025-12-07 08:50:16,842 E 6328 6376] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(pid=6502) Running 0: 0.00 row [00:00, ? row/s]

[SGDClassifier(alpha=8e-05),
 SGDClassifier(alpha=9e-05),
 SGDClassifier(),
 SGDClassifier(alpha=0.00011),
 SGDClassifier(alpha=0.00012)]

In [47]:
# Get validation results from each worker.
print(ray.get([worker.test.remote(X_test, Y_test) for worker in workers]))

ray.shutdown()

[0.68, 0.8, 0.68, 0.84, 0.64]


In [48]:
import ray
from ray.util.dask import enable_dask_on_ray

ray.init()  # Start or connect to Ray.
enable_dask_on_ray()  # Enable the Ray scheduler backend for Dask.

2025-12-07 08:50:38,897	INFO worker.py:2023 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/dask/config.py:787: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


In [49]:
import dask

df = dask.datasets.timeseries()
df = df[df.y > 0].groupby("name").x.std()
df.compute()  # Trigger the task graph to be evaluated.

(pid=gcs_server) [2025-12-07 08:51:02,565 E 7006 7006] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


,x
name,
Bob,0.576972
Dan,0.576158
Edith,0.577642
George,0.576294
Hannah,0.577740
Ingrid,0.575510
Laura,0.578950
Michael,0.576906
Quinn,0.576029


In [50]:
import ray
ds = ray.data.range(10000)

# Convert the Dataset to a Dask DataFrame.
df = ds.to_dask()
print(df.std().compute())  # -> 2886.89568

# Convert the Dask DataFrame back to a Dataset.
ds = ray.data.from_dask(df)
print(ds.std())  # -> 2886.89568

(dask:('getitem-547800b92bc79cab9616f27ec56aaebe', 1) pid=7129) [2025-12-07 08:51:20,552 E 7129 7248] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
2025-12-07 08:51:26,301	INFO logging.py:397 -- Registered dataset logger for dataset dataset_0_0
2025-12-07 08:51:26,312	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_0_0. Full logs are in /tmp/ray/session_2025-12-07_08-50-32_398468_1005/logs/ray-data
2025-12-07 08:51:26,313	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_0_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadRange]


Running 0: 0.00 row [00:00, ? row/s]

- ReadRange 1: 0.00 row [00:00, ? row/s]

2025-12-07 08:51:29,280	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_0_0 execution finished in 2.96 seconds
2025-12-07 08:51:29,873	INFO logging.py:397 -- Registered dataset logger for dataset dataset_3_0
2025-12-07 08:51:29,885	INFO hash_aggregate.py:180 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.1MiB, output=0.1MiB, total=0.2MiB, 
2025-12-07 08:51:29,894	INFO streaming_executor.py:174 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2025-12-07_08-50-32_398468_1005/logs/ray-data
2025-12-07 08:51:29,895	INFO streaming_executor.py:175 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]


id    2886.89568
dtype: float64


...

Shuffle 2:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

Aggregation 3:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

- limit=1 4: 0.00 row [00:00, ? row/s]

2025-12-07 08:51:43,600	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_3_0 execution finished in 13.70 seconds
2025-12-07 08:51:43,657	INFO util.py:257 -- Exiting prefetcher's background thread


2886.8956799071675
